![Week 1 Day 3](../assets/banners/w1d3.png)

# Week 1 Day 3 - Structured Output: Reliable JSON, Every Time

Yesterday you learned to think in tokens, watched temperature at work, and drove a chat model with `invoke`, `stream` and `batch`. Every single reply was **free text**.

Free text is lovely for people and miserable for software. If GradeWise asks "how many marks is Question 3 worth?" and gets back *"Question 3 carries four (4) marks in total, split between..."*, some code now has to dig the number out, and one day the model will write it differently and that code will break.

Today we fix this properly. You'll write an extraction prompt the way professionals do, define the exact shape you want with **Pydantic**, get the model to answer in that shape with **`with_structured_output`**, and finally build a small **self-correcting** loop that sends the model its own mistakes to fix. We'll practise on one real question from Emma's midterm: Question 3, our old friend the deadlock.

![Your 6-week roadmap](../assets/roadmap/week1.png)

### 🎬 Videos for this lab
| # | Lecture |
|:-:|---|
| 1 | Day 3 - Prompting for Extraction: System Prompts, Templates, Examples |
| 2 | Day 3 - Structured Output with Pydantic: Reliable JSON Every Time |
| 3 | Day 3 - Validation, Retries and Self-Correcting Extraction |

### 🎯 By the end of this lab you will
1. ✍️ Build an extraction prompt with `ChatPromptTemplate`: role, rules and a worked example
2. 🧱 Define a Pydantic model, validate data with it, and read its JSON schema
3. 🎯 Make a model answer in that exact shape with `with_structured_output`
4. 🔁 Check the *meaning* of an answer, and loop the problems back so the model fixes itself

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import json
import os
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError, model_validator

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.exam.parse import parse_pdf
from gradewise.models import chat_model

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
        print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(settings.describe_models())
print(datasets.describe())

## Part 1: Prompting for extraction

First, the raw material. We read the question paper and the marking scheme as Markdown (Day 4 explains how), then cut out the parts about Question 3 with a tiny helper.

In [ ]:
exam_dir = DATA_DIR / "exams" / "cs_midterm"
paper_text = parse_pdf(exam_dir / "question_paper.pdf").text
scheme_text = parse_pdf(exam_dir / "marking_scheme.pdf").text


def between(text: str, start: str, end: str) -> str:
    """The text from `start` up to (not including) `end`."""
    i = text.index(start)
    return text[i : text.index(end, i)].strip()


q3_paper = between(paper_text, "**Question 3**", "**Question 4**")
q3_scheme = between(scheme_text, "## **Question 3", "## **Question 4")

print(q3_paper, "\n\n" + "─" * 60 + "\n")
print(q3_scheme)

A good extraction prompt has four parts, and the **system** message carries the first three:

| Part | Example | Why |
|---|---|---|
| **Role** | "You are an exam-understanding assistant at Northbridge Institute" | sets context and tone |
| **Rules** | "Copy the question text exactly. NEVER invent marks." | each rule prevents a mistake models really make |
| **Output format** | "Reply with JSON with the keys ..." | so software can read it (Part 2 does this far better) |
| **Examples** | one worked input → output pair | showing beats telling, especially for formats and IDs |

`ChatPromptTemplate` turns this into a reusable object with `{placeholders}`. Examples go in as a fake earlier turn of the conversation: a human message with the input and an AI message with the perfect answer. This is called **few-shot prompting**.

In [ ]:
SYSTEM = """You are GradeWise's exam-understanding assistant at Northbridge Institute.
You read ONE question from an exam paper, plus its marking scheme, and describe it as JSON.
Rules:
1. Copy the question text exactly as printed, without the marks in brackets.
2. NEVER invent marks: every number must come from the documents.
3. Reply with JSON only, with the keys: number, text, max_marks, question_type."""

EXAMPLE_INPUT = "**Question 9**\n\nState the four ACID properties of a transaction. **[4 marks]**"
EXAMPLE_OUTPUT = json.dumps(
    {
        "number": "9",
        "text": "State the four ACID properties of a transaction.",
        "max_marks": 4,
        "question_type": "definition",
    }
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM),
        HumanMessage(EXAMPLE_INPUT),  # 💡 the few-shot example: a perfect answer to imitate
        AIMessage(EXAMPLE_OUTPUT),  # ⚠️ a message OBJECT, not ("ai", ...): its { } braces aren't placeholders
        ("human", "{question}"),
    ]
)

# 💡 Templates are offline: format them to see exactly what the model will receive
for message in prompt.format_messages(question=q3_paper):
    print(f"── {message.type.upper()} ──\n{message.content}\n")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">We're about to send this prompt and get free text back, then try <code>json.loads</code> on it. Predict: will it parse first time? Will <code>max_marks</code> be the number <code>4</code> or the string <code>"4"</code>? Will the model wrap the JSON in a <code>```json</code> code fence? Run it a few times.</span>
        </td>
    </tr>
</table>

In [ ]:
llm = chat_model(temperature=0)  # 💡 GradeWise's way to get the everyday model (Day 1, Part 6)

reply = (prompt | llm).invoke({"question": q3_paper})  # 💡 the | pipe chains Runnables: template → model
print(reply.text)

try:
    data = json.loads(reply.text)
    print("\n✅ Parsed:", data, "| max_marks is a", type(data.get("max_marks")).__name__)
except json.JSONDecodeError as err:
    print(f"\n❌ Not valid JSON: {err}")

It probably worked. But "probably" isn't good enough for 200 scripts: nothing *guarantees* valid JSON, the right keys, or a number where we need a number. Let's get guarantees.

## Part 2: Structured output with Pydantic

**Pydantic** lets you describe data as a Python class with type hints. It then *validates* incoming data against that description: it converts what it safely can (`"4"` → `4.0`) and raises a clear error for anything it can't.

Here's our first schema, for one exam question. Every `Field(description=...)` is documentation for humans **and** for the model, as you'll see in a moment.

In [ ]:
class ExtractedQuestion(BaseModel):
    """One question from an exam paper."""

    number: str = Field(description='The question number as printed, e.g. "3" or "4(b)"')
    text: str = Field(description="The full question text, exactly as printed, without the marks")
    max_marks: float = Field(gt=0, description="The marks shown in square brackets")
    question_type: Literal["definition", "short", "long", "compare", "numerical"] = Field(
        description="The kind of answer expected"
    )


good = ExtractedQuestion.model_validate(
    {"number": "3", "text": "What is a deadlock? ...", "max_marks": "4", "question_type": "definition"}
)
print(repr(good))
print("max_marks is now a", type(good.max_marks).__name__)  # 💡 "4" became 4.0

In [ ]:
# ⚠️ And here's what bad data looks like. Pydantic reports EVERY problem at once, with its location.
try:
    ExtractedQuestion.model_validate({"number": 3, "max_marks": -4, "question_type": "essay"})
except ValidationError as err:
    for e in err.errors():
        print(f"{'.'.join(map(str, e['loc'])):<14} {e['msg']}")

Did you spot that `number: 3` (an `int`) was rejected? Pydantic happily turns `"4"` into `4.0`, but it won't quietly turn an `int` into a `str`. Strictness like this is a feature.

### What the model actually sees

Behind the scenes, LangChain turns your class into a **JSON Schema** and hands it to the provider. The docstring and the field descriptions are all in there: they *are* prompts.

In [ ]:
print(json.dumps(ExtractedQuestion.model_json_schema(), indent=2))

### `with_structured_output`: the one line that matters

`llm.with_structured_output(ExtractedQuestion)` returns a new Runnable. You invoke it like any model, and instead of an `AIMessage` you get back **an `ExtractedQuestion` instance**, already validated. With OpenAI, LangChain uses the provider's native *structured outputs* mode, so the reply is constrained to the schema as it is generated. Notice the system prompt no longer needs to describe the JSON format.

In [ ]:
structured_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are GradeWise's exam-understanding assistant. Describe the question you are given. "
            "Copy the text exactly; never invent marks.",
        ),
        ("human", "{question}"),
    ]
)

extractor = structured_prompt | llm.with_structured_output(ExtractedQuestion)
q3 = extractor.invoke({"question": q3_paper})

print(type(q3).__name__)
print(q3.model_dump_json(indent=2))

In [ ]:
# 💡 include_raw=True gives you three things: the raw AIMessage, the parsed object, and any parsing error.
#    Handy for debugging, and for logging token usage.
debug = (structured_prompt | llm.with_structured_output(ExtractedQuestion, include_raw=True)).invoke(
    {"question": q3_paper}
)
print("parsed       :", debug["parsed"])
print("parsing_error:", debug["parsing_error"])
print("token usage  :", debug["raw"].usage_metadata)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Structured output guarantees the <b>shape</b>, not the <b>truth</b>. The model can return a perfectly valid <code>ExtractedQuestion</code> that says the question is worth 5 marks. Also: not every model supports native structured output (many small local models don't), and very long or deeply nested schemas are harder for a model to fill. Keep schemas as simple as the job allows, and describe every field.</span>
        </td>
    </tr>
</table>

## Part 3: Validation, retries and self-correcting extraction

There are two levels of "correct":

1. **Shape:** right fields, right types, numbers greater than zero. Pydantic checks this at parse time.
2. **Meaning:** do the marking criteria add up to the question's marks? Is every ID unique? These are *business rules*.

Let's add the marking criteria to our question. We'll write the meaning check twice, to see the trade-off: once as a Pydantic `model_validator` (it raises), and once as a `problems()` method (it returns a list).

In [ ]:
class Criterion(BaseModel):
    """One mark-earning point from the marking scheme."""

    description: str = Field(description="What the student must say or do, in the scheme's words")
    marks: float = Field(gt=0)


class QuestionWithCriteria(ExtractedQuestion):  # 💡 inherits number, text, max_marks, question_type
    criteria: list[Criterion] = Field(min_length=1, description="Every mark point in the scheme, in order")

    def problems(self) -> list[str]:
        """Meaning checks that DON'T raise: we want to send them back to the model."""
        total = sum(c.marks for c in self.criteria)
        if abs(total - self.max_marks) > 0.01:
            return [f"The criteria add up to {total:g} marks, but the question is worth {self.max_marks:g}."]
        return []


class StrictQuestion(QuestionWithCriteria):
    @model_validator(mode="after")
    def _criteria_add_up(self):
        if issues := self.problems():
            raise ValueError(issues[0])
        return self

In [ ]:
# 🆓 Offline: a "model answer" that forgot one of the deadlock conditions
forgetful = {
    "number": "3",
    "text": "What is a deadlock? Name TWO of the four necessary conditions for a deadlock.",
    "max_marks": 4,
    "question_type": "definition",
    "criteria": [
        {"description": "A correct definition of deadlock", "marks": 2},
        {"description": "Names a first correct necessary condition", "marks": 1},
    ],
}

soft = QuestionWithCriteria.model_validate(forgetful)
print("problems():", soft.problems())

try:
    StrictQuestion.model_validate(forgetful)
except ValidationError as err:
    print("StrictQuestion raised:", err.errors()[0]["msg"])

Both caught the mistake. So which is better? Think about what happens *inside* `with_structured_output`:

- With `StrictQuestion`, the validator raises while LangChain parses the reply. You get an exception, and the model's (mostly right!) answer is gone.
- With `problems()`, you keep the answer **and** a plain-English list of what's wrong. That list is exactly what you'd want to tell the model.

So GradeWise uses Pydantic for the *shape*, and a `problems()` list for the *meaning*. Then it **loops**: if there are problems, send the model its previous answer plus the problems, and ask for a fix. This is **self-correction**.

In [ ]:
FIX_REQUEST = "Your answer has these problems:\n{issues}\nRe-read the marking scheme and return the corrected answer."


def extract_with_feedback(llm, messages: list, max_attempts: int = 3) -> QuestionWithCriteria:
    """Extract, check problems(), and send them back until the answer is clean."""
    extractor = llm.with_structured_output(QuestionWithCriteria)
    history = list(messages)
    for attempt in range(1, max_attempts + 1):
        answer = extractor.invoke(history)
        issues = answer.problems()
        if not issues:
            print(f"✅ attempt {attempt}: clean")
            return answer
        print(f"⚠️ attempt {attempt}: {issues}")
        # 💡 show the model its own answer, then the list of problems
        history += [AIMessage(answer.model_dump_json()), HumanMessage(FIX_REQUEST.format(issues="\n".join(issues)))]
    print("🙋 still not clean: a human should check this one")
    return answer

### A dress rehearsal, for free

Before spending money, let's check the loop logic with a **pretend model**. It has a `with_structured_output` method like a real one, but just hands back canned answers: first the forgetful one, then a fixed one.

In [ ]:
from langchain_core.runnables import RunnableLambda


class PretendModel:
    """Stands in for a chat model: with_structured_output() returns the next canned answer."""

    def __init__(self, answers: list[dict]):
        self.answers = list(answers)

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(lambda messages: schema.model_validate(self.answers.pop(0)))


fixed = {**forgetful, "criteria": forgetful["criteria"] + [{"description": "Names a second condition", "marks": 1}]}

result = extract_with_feedback(PretendModel([forgetful, fixed]), messages=["(the prompt would go here)"])
print(f"{len(result.criteria)} criteria, adding up to {sum(c.marks for c in result.criteria):g}")

The loop works. Now the real thing: the question **and** its marking scheme, on a real model.

In [ ]:
messages = [
    (
        "system",
        "You are GradeWise's exam-understanding assistant. From the question and its marking scheme, "
        "extract the question and EVERY mark point as a criterion. Copy wording; never invent marks.",
    ),
    ("human", f"=== QUESTION ===\n{q3_paper}\n\n=== MARKING SCHEME ===\n{q3_scheme}"),
]

q3_full = extract_with_feedback(llm, messages)
for c in q3_full.criteria:
    print(f"  [{c.marks:g}] {c.description}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">This is exactly how the real extractor works. Open <code>gradewise/exam/extract.py</code>: <code>extract_once</code> calls <code>llm.with_structured_output(ExamPaper)</code> and returns <code>(paper, problems)</code>; <code>build_messages</code> adds the previous answer and the problem list on a retry; and <code>extract_with_retry</code> is our loop, grown up (it keeps the best attempt and asks a human when it gives up). The <code>problems()</code> method lives on <code>ExamPaper</code> in <code>gradewise/schemas/exam.py</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Self-correction is cheap insurance. A retry costs one extra call, only when something is wrong, and it turns "the model is right 90% of the time" into "the model is right, or a human is told". Clients care far more about the second sentence: it's the difference between a demo and a product they can trust.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- ✍️ wrote an extraction prompt with a role, rules and a few-shot example, using `ChatPromptTemplate`
- 🧱 defined Pydantic models, watched them coerce and reject data, and read the JSON Schema the model sees
- 🎯 used `with_structured_output` (and `include_raw=True`) to get validated objects, not text
- 🔁 split **shape** checks (Pydantic) from **meaning** checks (`problems()`), and built a self-correcting loop, rehearsed for free with a pretend model

So far the model has only read **text**. But exam scripts are scans of handwriting, and some questions come with diagrams. **Tomorrow (Day 4)** we send the model *images* of PDF pages, and give it **tools** it can call, like looking up how many marks a question is worth.

⏭ **Next:** `week1/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>Extract Question 5.</b> Question 5 (the paging calculation) has five mark points, some for <i>method</i>. (1) Cut out its question text and marking scheme with <code>between(...)</code>. (2) Add a <code>reference_topics: list[str]</code> field to a subclass of <code>QuestionWithCriteria</code>, with a good description. (3) Add a second check to <code>problems()</code>: every criterion description must be at least 10 characters. (4) Run <code>extract_with_feedback</code> on it, and rehearse first with <code>PretendModel</code>. Solution in <code>week1/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangChain structured output: <a href="https://docs.langchain.com/oss/python/langchain/structured-output">docs.langchain.com/oss/python/langchain/structured-output</a><br/> • Pydantic models and validators: <a href="https://docs.pydantic.dev/latest/concepts/models/">docs.pydantic.dev</a><br/> • Prompt templates: <a href="https://docs.langchain.com/oss/python/langchain/models">docs.langchain.com</a><br/> • New to type hints or classes? <code>guides/10_intermediate_python</code></span>
        </td>
    </tr>
</table>